In [1]:
# Local BraTS extraction with segmentation masks (pre-Colab step)
import os
import csv
import zipfile
import shutil
from pathlib import Path
import numpy as np
import nibabel as nib
from tqdm import tqdm
from skimage.transform import resize

# =========================
# CONFIG
# =========================
RAW_BRATS_DIR = Path(r"c:\Users\rifad\symAD-ECNN\data\brats2021")
OUT_DIR = Path(r"c:\Users\rifad\symAD-ECNN\data\brats_test_with_masks_raw")
ZIP_PATH = OUT_DIR.parent / "brats_test_with_masks_raw.zip"
TARGET_SIZE = (128, 128)
MIDDLE_PERCENTAGE = 0.60
SLICES_PER_PATIENT = 10
MIN_NONZERO_RATIO = 0.12

# Output structure
IMAGES_DIR = OUT_DIR / "images"
MASKS_DIR = OUT_DIR / "masks"
LABELS_DIR = OUT_DIR / "labels"
META_CSV = OUT_DIR / "metadata.csv"

# =========================
# HELPERS
# =========================
def normalize01(x: np.ndarray) -> np.ndarray:
    x = x.astype(np.float32)
    mn, mx = float(x.min()), float(x.max())
    if mx - mn < 1e-6:
        return np.zeros_like(x, dtype=np.float32)
    return (x - mn) / (mx - mn)

def is_valid_slice(img: np.ndarray, min_nonzero_ratio: float = MIN_NONZERO_RATIO) -> bool:
    return (np.count_nonzero(img) / img.size) >= min_nonzero_ratio

def safe_resize_image(img2d: np.ndarray, out_size=(128, 128)) -> np.ndarray:
    return resize(
        img2d, out_size,
        order=3,
        mode='reflect',
        anti_aliasing=True,
        preserve_range=True
    ).astype(np.float32)

def safe_resize_mask(mask2d: np.ndarray, out_size=(128, 128)) -> np.ndarray:
    # Nearest-neighbor to preserve discrete labels
    out = resize(
        mask2d, out_size,
        order=0,
        mode='constant',
        anti_aliasing=False,
        preserve_range=True
    )
    return (out > 0).astype(np.uint8)

def find_t1_and_seg(patient_path: Path):
    t1_path, seg_path = None, None
    for f in patient_path.glob("*.nii.gz"):
        name = f.name.lower()
        if name.endswith("_t1.nii.gz") and not name.endswith("_t1ce.nii.gz"):
            t1_path = f
        elif name.endswith("_seg.nii.gz"):
            seg_path = f
    return t1_path, seg_path

# =========================
# MAIN
# =========================
if not RAW_BRATS_DIR.exists():
    raise FileNotFoundError(f"BraTS folder not found: {RAW_BRATS_DIR}")

# Reset output
if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
IMAGES_DIR.mkdir(parents=True, exist_ok=True)
MASKS_DIR.mkdir(parents=True, exist_ok=True)
LABELS_DIR.mkdir(parents=True, exist_ok=True)

patient_dirs = sorted([p for p in RAW_BRATS_DIR.iterdir() if p.is_dir()])
print(f"Patients found: {len(patient_dirs)}")

global_idx = 0
rows = []
skipped = 0

for pdir in tqdm(patient_dirs, desc="Extracting slices"):
    t1_path, seg_path = find_t1_and_seg(pdir)
    if t1_path is None or seg_path is None:
        skipped += 1
        continue

    try:
        # Force canonical RAS orientation for both image and segmentation
        t1_vol = nib.as_closest_canonical(nib.load(str(t1_path))).get_fdata()
        seg_vol = nib.as_closest_canonical(nib.load(str(seg_path))).get_fdata()

        if t1_vol.shape[:3] != seg_vol.shape[:3]:
            skipped += 1
            continue

        z = t1_vol.shape[2]
        margin = int(z * (1 - MIDDLE_PERCENTAGE) / 2)
        start = max(0, margin)
        end = min(z, z - margin)
        if end <= start:
            skipped += 1
            continue

        indices = np.linspace(start, end - 1, SLICES_PER_PATIENT).astype(int)

        for s in indices:
            img_s = t1_vol[:, :, s]
            seg_s = seg_vol[:, :, s]

            if not is_valid_slice(img_s):
                continue

            img_res = safe_resize_image(img_s, TARGET_SIZE)
            img_res = normalize01(img_res)

            mask_res = safe_resize_mask(seg_s, TARGET_SIZE)
            label = 1 if int(mask_res.sum()) > 0 else 0

            base = f"slice_{global_idx:06d}"
            np.save(IMAGES_DIR / f"{base}.npy", img_res.astype(np.float32))
            np.save(MASKS_DIR / f"{base}.npy", mask_res.astype(np.uint8))
            np.save(LABELS_DIR / f"label_{global_idx:06d}.npy", np.array([label], dtype=np.uint8))

            rows.append([base, pdir.name, int(s), label, int(mask_res.sum())])
            global_idx += 1

    except Exception:
        skipped += 1
        continue

with open(META_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["slice_id", "patient_id", "z_index", "label", "mask_pixels"])
    writer.writerows(rows)

# Create zip for Drive upload
if ZIP_PATH.exists():
    ZIP_PATH.unlink()
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for sub in [IMAGES_DIR, MASKS_DIR, LABELS_DIR]:
        for fp in sorted(sub.glob("*.npy")):
            zf.write(fp, arcname=str(fp.relative_to(OUT_DIR)).replace("\\", "/"))
    zf.write(META_CSV, arcname="metadata.csv")

size_mb = ZIP_PATH.stat().st_size / (1024 * 1024)
print("\n=== EXTRACTION COMPLETE ===")
print(f"Total slices: {global_idx:,}")
print(f"Skipped patients/files: {skipped}")
print(f"Output folder: {OUT_DIR}")
print(f"ZIP ready: {ZIP_PATH}")
print(f"ZIP size: {size_mb:.1f} MB")
print("\nUpload this ZIP to Drive, e.g. /content/drive/MyDrive/symAD-ECNN/data/")

Patients found: 1251


Extracting slices: 100%|██████████| 1251/1251 [05:26<00:00,  3.83it/s]



=== EXTRACTION COMPLETE ===
Total slices: 11,529
Skipped patients/files: 0
Output folder: c:\Users\rifad\symAD-ECNN\data\brats_test_with_masks_raw
ZIP ready: c:\Users\rifad\symAD-ECNN\data\brats_test_with_masks_raw.zip
ZIP size: 394.1 MB

Upload this ZIP to Drive, e.g. /content/drive/MyDrive/symAD-ECNN/data/
